# 单期 Brinson 归因与报告

使用本地模拟 Dow 30 组合的一个月数据。本教程只调用**单期**计算和报告入口。收益率与效应采用十进制；图表会显示百分比或基点。

## 1. 读取本地数据

CSV 输入位于 `data/dow30_brinson/`。可从项目根目录或任一语言教程目录启动 Jupyter。

In [ ]:
from pathlib import Path
import sys
import polars as pl
import matplotlib.pyplot as plt

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "brinsons").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Launch Jupyter from the project root or tutorials/.")
sys.path.insert(0, str(ROOT / "src"))
from brinsons import (
    ColumnNames, bhb, fachler, calculate_single_period,
    print_single_period_category_weights, print_single_period_category_returns,
    print_single_period_category_attribution, print_single_period_total_attribution,
    print_single_period_attribution_summary, chart_single_period_category_returns,
    chart_single_period_category_weights, chart_single_period_category_attribution,
    chart_single_period_excess_return_waterfall,
)

asset = pl.read_csv(ROOT / "data" / "dow30_brinson" / "asset_table.csv")
periods = pl.read_csv(ROOT / "data" / "dow30_brinson" / "periods.csv")
print(f"{asset['Name'].n_unique()} stocks, {periods.height} monthly periods")
display(asset.head(6))

## 2. 汇总行业收益并保留原始列名

按期间和行业汇总个股权重及加权收益。`ColumnNames` 记录原始列名，计算时无需改名。

In [ ]:
sector = asset.group_by(["Period", "Category"]).agg(
    pl.col("PortfolioWeight", "BenchmarkWeight").sum(),
    (pl.col("PortfolioWeight") * pl.col("Return")).sum().alias("PortfolioContribution"),
    (pl.col("BenchmarkWeight") * pl.col("Return")).sum().alias("BenchmarkContribution"),
).with_columns(
    (pl.col("PortfolioContribution") / pl.col("PortfolioWeight")).alias("PortfolioReturn"),
    (pl.col("BenchmarkContribution") / pl.col("BenchmarkWeight")).alias("BenchmarkReturn"),
).sort(["Period", "Category"])
column_names = ColumnNames(period="Period", segment="Category",
    portfolio_weight="PortfolioWeight", benchmark_weight="BenchmarkWeight",
    portfolio_return="PortfolioReturn", benchmark_return="BenchmarkReturn")
display(sector.head())

## 3. 计算一个期间

显式选择第 1 期。Brinson–Fachler 三效应为
$A_i=(w_{P,i}-w_{B,i})(r_{B,i}-R_B)$、
$S_i=w_{B,i}(r_{P,i}-r_{B,i})$、
$I_i=(w_{P,i}-w_{B,i})(r_{P,i}-r_{B,i})$。
三个效应的合计必须等于 $R_P-R_B$。

In [ ]:
result = calculate_single_period(sector, period=1, attribution=fachler,
                                 effects=3, columns=column_names)
display(result.summary)
display(result.detail.head())

## 4. 比较 BHB/BF 与两效应/三效应

BHB 的配置基准是行业基准收益；BF 再减去整体基准收益。两效应版本把交互效应并入选股效应。

In [ ]:
single_method_rows = []
single_category_parts = []
for brinson_name, brinson_fn in [("BHB", bhb), ("BF", fachler)]:
    for effect_count in (2, 3):
        run = calculate_single_period(sector, period=1, attribution=brinson_fn,
                                      effects=effect_count, columns=column_names)
        names = run.columns
        totals = run.summary.row(0, named=True)
        single_method_rows.append({
            "Brinson": brinson_name, "Effects": effect_count,
            "Portfolio (bp)": 10_000 * totals[names.summary_portfolio_return],
            "Benchmark (bp)": 10_000 * totals[names.summary_benchmark_return],
            "Active (bp)": 10_000 * totals[names.summary_excess_return],
            "Allocation (bp)": 10_000 * totals[names.summary_allocation],
            "Selection (bp)": 10_000 * totals[names.summary_selection],
            "Interaction (bp)": 10_000 * totals[names.summary_interaction],
            "Residual (bp)": 10_000 * (
                totals[names.summary_allocation] + totals[names.summary_selection]
                + totals[names.summary_interaction] - totals[names.summary_excess_return]),
        })
        single_category_parts.append(run.detail.select(
            pl.lit(brinson_name).alias("Brinson"),
            pl.lit(effect_count).alias("Effects"),
            pl.col(names.segment).alias("Category"),
            (10_000 * pl.col(names.allocation)).alias("Allocation (bp)"),
            (10_000 * pl.col(names.selection)).alias("Selection (bp)"),
            (10_000 * pl.col(names.interaction)).alias("Interaction (bp)"),
        ))

single_method_comparison = pl.DataFrame(single_method_rows)
single_category_comparison = pl.concat(single_category_parts).sort(["Category", "Brinson", "Effects"])
assert single_method_comparison["Residual (bp)"].abs().max() < 1e-6
with pl.Config(tbl_rows=100, tbl_cols=12, tbl_width_chars=180, fmt_str_lengths=32):
    display(single_method_comparison)
    display(single_category_comparison)


### 行业配置效应对比

总配置效应可能相同，但各行业的配置效应不同。

In [ ]:
bhb_allocation = single_category_comparison.filter(
    (pl.col("Brinson") == "BHB") & (pl.col("Effects") == 3)).sort("Category")
bf_allocation = single_category_comparison.filter(
    (pl.col("Brinson") == "BF") & (pl.col("Effects") == 3)).sort("Category")
categories = bhb_allocation["Category"].to_list()
assert categories == bf_allocation["Category"].to_list()
positions = list(range(len(categories)))
fig, ax = plt.subplots(figsize=(10, max(4, 0.42 * len(categories))), layout="constrained")
ax.barh([p - 0.18 for p in positions], bhb_allocation["Allocation (bp)"].to_list(),
        height=0.34, label="BHB", color="#2874a6")
ax.barh([p + 0.18 for p in positions], bf_allocation["Allocation (bp)"].to_list(),
        height=0.34, label="BF", color="#d88935")
ax.set_yticks(positions, categories)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Allocation effect (bp)")
ax.set_title("Same period and sectors: BHB vs BF allocation")
ax.legend()
plt.show()


## 5. 行业权重

以下函数都显式传入单期 `period=1`。

In [ ]:
print_single_period_category_weights(result, period=1);

## 6. 行业收益

In [ ]:
print_single_period_category_returns(result, period=1);

## 7. 行业归因

In [ ]:
print_single_period_category_attribution(result, period=1);

## 8. 总归因

In [ ]:
print_single_period_total_attribution(result, period=1);

## 9. 绩效归因摘要

In [ ]:
print_single_period_attribution_summary(result, period=1);

## 10. 横向柱状图：收益

In [ ]:
fig, ax = chart_single_period_category_returns(result, period=1)
plt.show()

## 11. 横向柱状图：权重

In [ ]:
fig, ax = chart_single_period_category_weights(result, period=1)
plt.show()

## 12. 横向柱状图：归因效应

In [ ]:
fig, ax = chart_single_period_category_attribution(result, period=1)
plt.show()

## 13. 超额收益瀑布图

In [ ]:
fig, ax = chart_single_period_excess_return_waterfall(result, period=1)
plt.show()

## 14. 交互查看其他期间

滑块改变时会**重新计算所选单期**，不会调用多期链接方法。

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def show_period(period):
    selected = calculate_single_period(sector, period=period, attribution=fachler,
                                       columns=column_names)
    print_single_period_category_weights(selected, period=period)
    print_single_period_category_attribution(selected, period=period)
    fig, ax = chart_single_period_category_attribution(selected, period=period)
    plt.show()

slider = widgets.IntSlider(value=1, min=int(periods["Period"].min()),
                           max=int(periods["Period"].max()), description="Period",
                           continuous_update=False)
output_widget = widgets.interactive_output(show_period, {"period": slider})
display(slider, output_widget)

行业数据与组合权重仅供演示。单期超额收益是算术差；复合区间归因请阅读独立的多期教程。